In [1]:
#setting up rag

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
#load the data

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
#create the assistant

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [10]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — but it depends on the course’s enrollment rules and whether it’s still open.\n\nQuick things to check:\n- **Enrollment deadline:** Is registration still open?\n- **Prerequisites:** Do you meet any required background or prior courses?\n- **Capacity:** Is there still a seat available?\n- **Course format:** Some courses allow late joining; others don’t.\n\nIf you want, I can help you draft a short message to the instructor or registrar asking if you can still enroll.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [31]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseOutputMessage(id='msg_04384ee81f115576006a1d6e0d4d3c8199b4065c6f561d074e', content=[ResponseOutputText(annotations=[], text='Yes, you can still join and start learning.\n\nIf you want to receive a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [15]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [23]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
})

In [27]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join join course discovered"}', call_id='call_CzbXpmaSOk2iKABFSzUn96bR', name='search', type='function_call', id='fc_04384ee81f115576006a1d6c8aafd48199b98495fc2f5facdb', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_CzbXpmaSOk2iKABFSzUn96bR',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. W

In [33]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still open.'

In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(658, 34)

In [35]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [37]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [38]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join late enrollment FAQ"}
function_call: search {"query":"course registration enrollment late join FAQ discovered course"}
function_call: search {"query":"can I join the course after it started FAQ"}


In [39]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

A couple of things to keep in mind:
- You can start learning and submitting homework while the submission form is open.
- If you want a certificate, you need to submit your project while submissions are still being accepted.
- Certificates are only available for the live cohort, not self-paced.

If you want, I can also help with what to do next after joining.


In [40]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [41]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run Ollama locally install run local model FAQ"}
iteration #2...
function_call: search {"query":"Ollama localhost 11434 ollama run llama3 python client FAQ local server"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat interface.

3. **Verify the local server is running**
   ```bash
   curl http://localhost:11434
   ```

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you get a connection i

'To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat interface.\n\n3. **Verify the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server with:\n```bash\nollama serve\n```\ncan help.\n\nIf you want, I can also show you how to run Ollama with a different model or how to connect it to a Python notebook.'

In [42]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"Can I still join the course late enrollment discovered course join late FAQ"}
function_call: search {"query":"course enrollment late join discovered after start FAQ"}
function_call: search {"query":"joining the course after it has started enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.

If you'd like, I can also help with what you need to do next to get started. Are there other areas you want to explore?


"Yes — you can still join the course.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still being accepted.\n\nIf you'd like, I can also help with what you need to do next to get started. Are there other areas you want to explore?"

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join late enrollment FAQ"}
iteration #2...
function_call: search {"query":"certificate live cohort submit project while accepting submissions peer-review course running self-paced mode FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted, and certificates are only available if you finish with the live cohort.

If you’d like, I can also explain what “joining late” means for homework, project submission, and certificates.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted, and certificates are only available if you finish with the live cohort.\n\nIf you’d like, I can also explain what “joining late” means for homework, project submission, and certificates.'

In [44]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit definition"}
iteration #2...
function_call: search {"query":"queen gambit chess opening what is queen's gambit"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a chess opening.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to distract Black’s d-pawn and gain control of the center. It’s called a “gambit” because White is offering a pawn, at least temporarily.

There are two main ideas:
- **Queen’s Gambit Accepted**: Black takes the c-pawn.
- **Queen’s Gambit Declined**: Black does not take it.

If you want, I can also explain:
- the **basic plan** for White and Black,
- the difference between **accepted vs declined**,
- or show a **simple move-by-move example**.


'A **Queen’s Gambit** is a chess opening.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to distract Black’s d-pawn and gain control of the center. It’s called a “gambit” because White is offering a pawn, at least temporarily.\n\nThere are two main ideas:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn.\n- **Queen’s Gambit Declined**: Black does not take it.\n\nIf you want, I can also explain:\n- the **basic plan** for White and Black,\n- the difference between **accepted vs declined**,\n- or show a **simple move-by-move example**.'

In [45]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening what is queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this isn’t a course-related question.

If you meant “Queen’s Gambit” in chess, I can’t answer that from the course FAQ database. If you want, ask me about something course-related, and I’ll help with that. Are there other areas that you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this isn’t a course-related question.\n\nIf you meant “Queen’s Gambit” in chess, I can’t answer that from the course FAQ database. If you want, ask me about something course-related, and I’ll help with that. Are there other areas that you want to explore?'

In [48]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [50]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [51]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [52]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [53]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [54]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [55]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [58]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [59]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='hello', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"hello"}', call_id='call_UxVcYqSOMXTwYRdvg6SkO4nO', name='searc

In [60]:
result.cost

CostInfo(input_cost=Decimal('0.002391'), output_cost=Decimal('0.0013185'), total_cost=Decimal('0.0037095'))